# 📦 Ejercicios: Walmart
# Cadena de suministro: stock, roturas y lead time

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de Walmart.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué producto sufre
### más roturas de
### stock? (ranking)

Rotura = venta perdida. Priorizamos
los productos que más se agotan.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/walmart_inventario.csv'
)

r = (
  df
  .groupby('producto')
  .agg(
    roturas=('roturas_stock',
             'sum'),
    stock=('stock', 'mean'),
  )
  .round(0)
  .sort_values('roturas',
    ascending=False)
)

print('Roturas por producto:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿El lead time largo
### provoca más
### roturas? (corr)

Si el proveedor tarda, el stock no
llega a tiempo. Medimos la relación.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/walmart_inventario.csv'
)

corr = (
  df['lead_time']
  .corr(df['roturas_stock'])
)

df['tramo'] = pd.cut(
  df['lead_time'],
  bins=[0, 5, 10, 100],
  labels=['Rápido', 'Medio',
          'Lento'],
)

r = (
  df
  .groupby('tramo', observed=True)
  ['roturas_stock']
  .mean()
  .round(2)
)

print(f'Correlación: {corr:.2f}')
print('\nRoturas medias por lead:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué proveedor es
### más fiable y más
### barato a la vez?

Cruzamos lead time, roturas y coste
para elegir proveedor estratégico.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/walmart_inventario.csv'
)

r = (
  df
  .groupby('proveedor')
  .agg(
    lead=('lead_time', 'mean'),
    roturas=('roturas_stock',
             'mean'),
    coste=('coste_usd', 'mean'),
  )
  .round(2)
  .sort_values('roturas')
)

print('Fiabilidad por proveedor:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué almacén tiene
### más riesgo de
### quiebre? (índice)

Índice = roturas por cada 100 de
stock. Normaliza por tamaño.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/walmart_inventario.csv'
)

g = (
  df
  .groupby('almacen')
  .agg(
    roturas=('roturas_stock',
             'sum'),
    stock=('stock', 'sum'),
  )
)

g['indice'] = (
  g['roturas'] / g['stock'] * 100
).round(3)

print('Riesgo de quiebre x almacén:')
print(
  g['indice']
  .sort_values(ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Pocos productos
### concentran casi
### todas las roturas?

Pareto: ¿el 20% de productos causa
el 80% de las roturas de stock?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/walmart_inventario.csv'
)

r = (
  df
  .groupby('producto')
  ['roturas_stock']
  .sum()
  .sort_values(ascending=False)
)

acum = r.cumsum() / r.sum() * 100

tabla = pd.DataFrame({
  'roturas': r,
  'acum_%': acum.round(1),
})

print('Pareto de roturas:')
print(tabla)